# このノートブックの使い方

## 1. Google Colab とは
Google Colab（コラボ）は、**ブラウザ上で Python を実行できるノートブック環境**です。  
ソフトを自分のPCにインストールしなくても使えます。
Colab には主に2種類のセルがあります。
- **コードセル**：Python のコードを書く場所
- **テキストセル**：説明を書く場所（Markdown 形式）
---
## 2. 最初にすること
1. Google Colab で新しいノートブックを開く  
2. 左の**フォルダ**アイコンをクリックする  
3. 実験データの **Excel ファイル（.xlsx）** をアップロードする  
4. 下のコードを実行する
---
## 3. セルの使い方
- **コードを実行する**  
  セルの左の ▶ ボタンを押します。  
  または **Shift + Enter** でも実行できます。
- **新しいコードセルを追加する**  
  **「+ コード」** を押します。
- **説明を書くセルを追加する**  
  **「+ テキスト」** を押します。  
  このセルは Markdown 形式で書けます。
---
## 4. このコードで何をするか
このコードは、Excel ファイルに入っている実験データを読み込み、
- 層流と乱流のデータ整理
- 圧力損失やレイノルズ数の計算
- フィット計算
- グラフの作成
- 300 dpi の PNG 画像保存
を自動で行います。
---
## 5. 実行の流れ
1. Excel ファイルをアップロードする  
2. コードセルを上から順に実行する  
3. グラフが表示される  
4. `Plotting for Experiments 1&2.png` が保存される
---
## 6. 保存した画像をダウンロードする方法
グラフは `Plotting for Experiments 1&2.png` という名前で保存されます。  
ダウンロード方法は次のどちらかです。

### 方法A：左のファイル一覧からダウンロード
左のフォルダを開き、`Plotting for Experiments 1&2.png` を右クリックしてダウンロードします。
### 方法B：コードで自動ダウンロード
必要なら、コードの最後に次を追加します。
```python
from google.colab import files
files.download("Plotting for Experiments 1&2.png")

In [ ]:
!pip install -q japanize-matplotlib openpyxl scipy

# ================================================================
# 必要ライブラリ
# ================================================================
from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)

# 日本語フォント設定
try:
    import japanize_matplotlib
    japanize_matplotlib.japanize()
except ImportError:
    print("⚠️ japanize_matplotlib が見つかりません。日本語表示が崩れる場合があります。")

# ================================================================
# グローバルプロット設定
# ================================================================
plt.rcParams.update({
    "figure.dpi": 150,       # notebook 表示用
    "savefig.dpi": 300,      # 保存用
    "axes.linewidth": 1.0,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "lines.linewidth": 1.6,
    "lines.markersize": 7,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "legend.framealpha": 0.9,
    "figure.constrained_layout.use": True,
    "text.usetex": False,
})

COLORS = {
    "lam": "#4C78A8",
    "turb": "#C0392B",
    "fit_lam": "#4C78A8",
    "fit_turb": "#C0392B",
    "theory": "#C0392B",
    "exp": "#72B7B2",
    "ref": "purple",
}

# ================================================================
# ユーティリティ関数
# ================================================================
def find_excel_file():
    """Colab(/content) とカレントディレクトリから Excel を自動検出"""
    search_dirs = [Path("/content"), Path.cwd()]
    candidates = []

    for base in search_dirs:
        if base.exists():
            candidates.extend(base.glob("*流動*.xlsx"))
            candidates.extend(base.glob("*.xlsx"))

    unique = []
    seen = set()
    for p in candidates:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)

    if not unique:
        raise FileNotFoundError("Excel ファイルが見つかりません。Colab に .xlsx をアップロードしてください。")

    excel_path = str(unique[0])
    print(f"✅ 使用する Excel ファイル: {excel_path}")
    return excel_path


def read_cells_as_array(ws, col, row_start, row_end):
    """指定列・範囲を float の NumPy 配列で返す"""
    values = [ws[f"{col}{r}"].value for r in range(row_start, row_end + 1)]
    if any(v is None for v in values):
        raise ValueError(f"{ws.title}: {col}{row_start}:{col}{row_end} に空セルがあります。")
    return np.array(values, dtype=float)


def power_law_fit(x, y):
    """y = a x^n を log-log 線形回帰でフィット"""
    mask = (x > 0) & (y > 0)
    if mask.sum() < 2:
        raise ValueError("フィットに必要な正のデータ点が不足しています。")

    logx = np.log(x[mask])
    logy = np.log(y[mask])
    slope, intercept, r_value, _, _ = stats.linregress(logx, logy)

    return {
        "n": slope,
        "a": np.exp(intercept),
        "r2": r_value**2,
    }


def calc_section(Q_meas, dh_cm, D, L, rho_air, mu_air, rho_water, g,
                 sigma_Q_abs, sigma_dh_abs, q_unit):
    """
    流量・差圧データから v, ΔP, Re, f と誤差を計算
    q_unit: 'mL/s' or 'm3/h'
    """
    if q_unit == "mL/s":
        Q_m3s = Q_meas * 1e-6
    elif q_unit == "m3/h":
        Q_m3s = Q_meas / 3600.0
    else:
        raise ValueError("q_unit は 'mL/s' または 'm3/h' を指定してください。")

    dh_m = dh_cm * 1e-2

    v = 4 * Q_m3s / (np.pi * D**2)
    dP = rho_water * g * dh_m
    Re = rho_air * v * D / mu_air
    f_exp = dP / (2 * rho_air * v**2) * (D / L)

    # 点ごとの相対誤差
    sigma_Q_rel = sigma_Q_abs / Q_meas
    sigma_dP_rel = sigma_dh_abs / dh_cm

    # 誤差伝播
    sigma_v = v * sigma_Q_rel
    sigma_Re = Re * sigma_Q_rel
    sigma_f_rel = np.sqrt(sigma_dP_rel**2 + (2 * sigma_Q_rel)**2)
    sigma_f = f_exp * sigma_f_rel
    sigma_dP = dP * sigma_dP_rel

    return {
        "Q_meas": Q_meas,
        "Q_m3s": Q_m3s,
        "dh_cm": dh_cm,
        "dh_m": dh_m,
        "v": v,
        "dP": dP,
        "Re": Re,
        "f_exp": f_exp,
        "sigma_Q_rel": sigma_Q_rel,
        "sigma_dP_rel": sigma_dP_rel,
        "sigma_v": sigma_v,
        "sigma_dP": sigma_dP,
        "sigma_Re": sigma_Re,
        "sigma_f": sigma_f,
    }

# ================================================================
# Excel workbook 読み込み
# ================================================================
excel_path = find_excel_file()
wb = load_workbook(excel_path, data_only=True)

ws_lam = wb["Lab-laminar-flow"]
ws_turb = wb["Lab-turbulent-flow"]

# ================================================================
# 共通物性
# ================================================================
rho_air   = float(ws_turb["B8"].value)
mu_air    = float(ws_turb["C8"].value)
rho_water = float(ws_turb["B9"].value)
g         = float(ws_turb["E8"].value)

# ================================================================
# 実験 1（層流）
# ================================================================
D_lam = float(ws_lam["A22"].value)
L_lam = float(ws_lam["B22"].value)

dh_lam = read_cells_as_array(ws_lam, "B", 26, 30)      # [cm]
Q_lam_meas = read_cells_as_array(ws_lam, "D", 26, 30)  # [mL/s]

# Lab-laminar-flow: σ_h = B32, σ_Q = D32
sigma_dh_lam = float(ws_lam["B32"].value)   # [cm]
sigma_Q_lam  = float(ws_lam["D32"].value)   # [mL/s]

lam = calc_section(
    Q_meas=Q_lam_meas,
    dh_cm=dh_lam,
    D=D_lam,
    L=L_lam,
    rho_air=rho_air,
    mu_air=mu_air,
    rho_water=rho_water,
    g=g,
    sigma_Q_abs=sigma_Q_lam,
    sigma_dh_abs=sigma_dh_lam,
    q_unit="mL/s",
)

# ================================================================
# 実験 2（乱流）
# ================================================================
D_turb = float(ws_turb["A22"].value)
L_turb = float(ws_turb["B22"].value)

Q_target_turb = read_cells_as_array(ws_turb, "A", 26, 31)  # [m^3/h] target
Q_turb_meas   = read_cells_as_array(ws_turb, "B", 26, 31)  # [m^3/h]
dh_turb       = read_cells_as_array(ws_turb, "D", 26, 31)  # [cm]

# Lab-turbulent-flow: σ_Q = B33, σ_h = D33
sigma_Q_turb  = float(ws_turb["B33"].value)   # [m^3/h]
sigma_dh_turb = float(ws_turb["D33"].value)   # [cm]

turb = calc_section(
    Q_meas=Q_turb_meas,
    dh_cm=dh_turb,
    D=D_turb,
    L=L_turb,
    rho_air=rho_air,
    mu_air=mu_air,
    rho_water=rho_water,
    g=g,
    sigma_Q_abs=sigma_Q_turb,
    sigma_dh_abs=sigma_dh_turb,
    q_unit="m3/h",
)

# 理論式
turb["f_blasius"] = 0.0791 / turb["Re"]**0.25

# ================================================================
# フィット
# ================================================================
fit_lam = power_law_fit(lam["v"], lam["dP"])
fit_turb = power_law_fit(turb["v"], turb["dP"])

v_lam_fit = np.linspace(lam["v"].min() * 0.85, lam["v"].max() * 1.10, 200)
dP_lam_fit = fit_lam["a"] * v_lam_fit**fit_lam["n"]

v_turb_fit = np.linspace(turb["v"].min() * 0.85, turb["v"].max() * 1.10, 200)
dP_turb_fit = fit_turb["a"] * v_turb_fit**fit_turb["n"]

# ================================================================
# 作図
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# 共通の見た目を改善（サイズ・ラベルは変更しない）
for ax in axes:
    ax.set_axisbelow(True)
    ax.minorticks_on()
    ax.tick_params(axis="both", which="major", direction="in", length=5, width=1.0)
    ax.tick_params(axis="both", which="minor", direction="in", length=3, width=0.8)
    ax.grid(True, which="major", linestyle="--", linewidth=0.7, alpha=0.30)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.18)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

# ------------------------------------------------
# (a) ΔP vs <v>
# ------------------------------------------------
ax = axes[0]

ax.errorbar(
    lam["v"], lam["dP"],
    xerr=lam["sigma_v"], yerr=lam["sigma_dP"],
    fmt="s",
    color=COLORS["lam"],
    capsize=4,
    elinewidth=1.0,
    capthick=1.0,
    markeredgecolor="white",
    markeredgewidth=0.9,
    markersize=7.5,
    zorder=3,
    label="層流データ"
)
ax.errorbar(
    turb["v"], turb["dP"],
    xerr=turb["sigma_v"], yerr=turb["sigma_dP"],
    fmt="o",
    color=COLORS["turb"],
    capsize=4,
    elinewidth=1.0,
    capthick=1.0,
    markeredgecolor="white",
    markeredgewidth=0.9,
    markersize=7.5,
    zorder=3,
    label="乱流データ"
)

ax.plot(
    v_lam_fit, dP_lam_fit, "--",
    color=COLORS["fit_lam"],
    linewidth=2.0,
    dash_capstyle="round",
    zorder=2
)
ax.plot(
    v_turb_fit, dP_turb_fit, "--",
    color=COLORS["fit_turb"],
    linewidth=2.0,
    dash_capstyle="round",
    zorder=2
)

ax.set_xlabel(r"断面平均速度 $\langle v \rangle$ [m s$^{-1}$]")
ax.set_ylabel(r"圧力損失 Δ$P$ [Pa]")
ax.set_title("(a) 圧力損失 vs 断面平均速度")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

leg = ax.legend(fontsize=8)
leg.get_frame().set_edgecolor("0.8")
leg.get_frame().set_linewidth(0.8)

ax.text(
    0.55, 0.85,
    f"層流: $\\Delta P \\propto \\langle v \\rangle^{{{fit_lam['n']:.2f}}}$, $R^2 = {fit_lam['r2']:.3f}$\n"
    f"乱流: $\\Delta P \\propto \\langle v \\rangle^{{{fit_turb['n']:.2f}}}$, $R^2 = {fit_turb['r2']:.3f}$",
    transform=ax.transAxes, va="top", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="#FFF3CD", edgecolor="0.75", alpha=0.92)
)

# ------------------------------------------------
# (b) Moody 図: f vs Re
# ------------------------------------------------
ax = axes[1]

Re_all = np.logspace(1.5, 5, 800)
f_lam_theory = 16 / Re_all

Re_turb_theory = np.logspace(3, 5, 500)
f_blasius_plot = 0.0791 / Re_turb_theory**0.25

# 遷移域
ax.axvspan(2100, 3000, color="gray", alpha=0.08, zorder=0)

ax.loglog(
    Re_all, f_lam_theory, "--",
    color=COLORS["lam"],
    alpha=0.85,
    linewidth=1.8,
    dash_capstyle="round",
    label="層流理論: f = 16/Re"
)
ax.loglog(
    Re_turb_theory, f_blasius_plot, "--",
    color=COLORS["theory"],
    alpha=0.90,
    linewidth=1.8,
    dash_capstyle="round",
    label="乱流理論: Blasius 式"
)

ax.errorbar(
    lam["Re"], lam["f_exp"],
    xerr=lam["sigma_Re"], yerr=lam["sigma_f"],
    fmt="s",
    color=COLORS["lam"],
    capsize=4,
    elinewidth=1.0,
    capthick=1.0,
    markeredgecolor="white",
    markeredgewidth=0.9,
    markersize=7.5,
    zorder=5,
    label="層流データ"
)
ax.errorbar(
    turb["Re"], turb["f_exp"],
    xerr=turb["sigma_Re"], yerr=turb["sigma_f"],
    fmt="s",
    color=COLORS["turb"],
    capsize=4,
    elinewidth=1.0,
    capthick=1.0,
    markeredgecolor="white",
    markeredgewidth=0.9,
    markersize=7.5,
    zorder=5,
    label="乱流データ"
)

ax.axvline(2100, color="gray", linestyle=":", alpha=0.6, lw=1)
ax.axvline(3000, color="gray", linestyle=":", alpha=0.6, lw=1)
ax.text(1600, 0.03, " Re = 2100", fontsize=8, color="gray", rotation=90)
ax.text(3200, 0.03, " Re = 3000", fontsize=8, color="gray", rotation=90)

ax.set_xlabel("レイノルズ数 Re [–]")
ax.set_ylabel(r"摩擦係数 $f$ [–]")
ax.set_title("(b) 摩擦係数 vs Re")
ax.set_xlim(1e1, 1e5)
ax.set_ylim(1e-3, 1)

leg = ax.legend(fontsize=8, loc="upper right")
leg.get_frame().set_edgecolor("0.8")
leg.get_frame().set_linewidth(0.8)

# ------------------------------------------------
# 保存（300 dpi PNG）
# ------------------------------------------------
output_png = Path("Plotting for Experiments 1&2.png")
fig.savefig(
    output_png,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
    edgecolor="none"
)

plt.show()
# print(f"✅ 300 dpi PNG saved: {output_png.resolve()}")

# Colab ならそのままダウンロード
# try:
#     from google.colab import files
#     files.download(str(output_png))
# except Exception:
#     print("Colab 以外では自動ダウンロードしません。上の保存先から取得してください。")

# ================================================================
# 結果表示
# ================================================================
print("=== フィット結果 ===")
print(f"層流:  ΔP ∝ v^{fit_lam['n']:.3f}   (R² = {fit_lam['r2']:.4f}, 理論値 1.00)")
print(f"乱流:  ΔP ∝ v^{fit_turb['n']:.3f}   (R² = {fit_turb['r2']:.4f}, 理論値 1.75)")
print()
print("=== Re 範囲 ===")
print(f"層流: {lam['Re'].min():.0f} – {lam['Re'].max():.0f}")
print(f"乱流: {turb['Re'].min():.0f} – {turb['Re'].max():.0f}")